# 04. Trajectory Analysis (PAGA + DPT / scVelo)

**목적**: 세포 분화/발달 경로 분석

BAM 파일 또는 loom 파일 유무에 따라 자동으로 분기:
- `scvelo_ready`: scVelo RNA velocity (방향성 화살표)
- `bam_available`: velocyto 실행 안내 후 loom 생성 시 자동 진행
- `no_velocity`: PAGA + DPT + CytoTRACE

**입력**: `output/checkpoints/{dataset}/08_annotated.h5ad`  
**출력**: `output/figures/trajectory/{dataset}/`, `output/checkpoints/{dataset}/09_trajectory.h5ad`

In [ ]:
import sys
sys.path.insert(0, "../../")
sys.path.insert(0, "../")

import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
import yaml

from modules.io import load_adata, save_adata, detect_velocity_mode
from modules.plotting import umap_panel

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white")

In [ ]:
DATASET = "human_genes_only"
CHECKPOINT_DIR = f"../../output/checkpoints/{DATASET}"
FIG_OUT_DIR = Path(f"../../output/figures/trajectory/{DATASET}")
FIG_OUT_DIR.mkdir(parents=True, exist_ok=True)

with open("../../configs/samples.yaml") as f:
    samples_cfg = yaml.safe_load(f)

with open("../../configs/pipeline.yaml") as f:
    pipeline_cfg = yaml.safe_load(f)

velocity_cfg = samples_cfg.get("velocity", {})
pipe_vel_cfg = pipeline_cfg.get("velocity", {})

bam_dir  = velocity_cfg.get("bam_dir", "") or "../../data"
loom_dir = velocity_cfg.get("loom_dir", "")

# Snakemake 자동 생성 loom 경로도 확인
snakemake_loom = f"../../output/velocity/loom/{DATASET}/merged.loom"
if not loom_dir and Path(snakemake_loom).exists():
    loom_dir = str(Path(snakemake_loom).parent)

adata = load_adata(f"{CHECKPOINT_DIR}/08_annotated.h5ad")
print(adata)

## 0. Velocity 모드 감지

In [ ]:
velocity_mode = detect_velocity_mode("../../data")

# Snakemake output도 확인
if velocity_mode == "no_velocity":
    if Path(snakemake_loom).exists():
        velocity_mode = "scvelo_ready"
        loom_dir = str(Path(snakemake_loom).parent)

print(f"Detected velocity mode: {velocity_mode}")
print({
    "scvelo_ready":  "loom 또는 spliced layer 발견 → scVelo 실행",
    "bam_available": "BAM 파일 발견 → velocyto 실행 필요",
    "no_velocity":   "velocity 데이터 없음 → PAGA + DPT + CytoTRACE",
}.get(velocity_mode, "unknown"))

## 1. PAGA (모든 경로 공통)

In [ ]:
groupby = "cell_type" if "cell_type" in adata.obs.columns else "leiden"

sc.tl.paga(adata, groups=groupby)
sc.pl.paga(
    adata,
    color=groupby,
    title="PAGA graph",
    save=None,
)

In [ ]:
sc.tl.draw_graph(adata, init_pos="paga")
sc.pl.draw_graph(adata, color=groupby, legend_loc="right margin")

## 2A. velocyto 실행 안내 (BAM 파일 있는 경우)

In [ ]:
if velocity_mode == "bam_available":
    bam_cfg = samples_cfg.get("velocity", {})
    gtf_path = pipe_vel_cfg.get("gtf", "<GTF_PATH>")
    
    print("=" * 60)
    print("BAM 파일 감지됨. 다음 단계로 velocyto를 실행하세요.")
    print("=" * 60)
    print()
    print("【방법 1】 Snakemake 자동 실행 (권장)")
    print(f"""
  # configs/pipeline.yaml 에서 velocity.gtf 경로 설정 후:
  snakemake --snakefile src/Snakefile --cores 8 \\
    output/velocity/loom/{DATASET}/merged.loom
""")
    print()
    print("【방법 2】 수동 실행 (샘플별)")
    print(f"""
  # 1. BAM 정렬 및 인덱싱
  samtools sort -@ 4 -o sample.sorted.bam sample.bam
  samtools index sample.sorted.bam

  # 2. 바코드 추출 (h5ad에서)
  python -c "
  import scanpy as sc
  adata = sc.read_h5ad('{CHECKPOINT_DIR}/08_annotated.h5ad')
  for s in adata.obs['sample'].unique():
      bc = adata.obs_names[adata.obs['sample']==s].tolist()
      open(f'{{s}}_barcodes.tsv','w').write('\\n'.join(bc))
  "

  # 3. velocyto 실행
  velocyto run \\
      -b <sample>_barcodes.tsv \\
      -o output/velocity/loom/{DATASET}/ \\
      <sample>.sorted.bam \\
      {gtf_path}
""")
    print()
    print("velocyto 실행 완료 후 이 노트북을 다시 실행하면 자동으로 scVelo가 진행됩니다.")

## 2B. scVelo RNA Velocity (loom 파일 있는 경우)

In [ ]:
if velocity_mode == "scvelo_ready":
    try:
        import scvelo as scv
    except ImportError:
        raise ImportError("scvelo가 설치되지 않았습니다. 설치: pip install scvelo")

    scv.settings.verbosity = 2
    scv.settings.set_figure_params("scvelo", dpi=100)
    scv_mode = pipe_vel_cfg.get("scvelo_mode", "dynamical")
    n_top_genes = pipe_vel_cfg.get("n_top_genes", 2000)
    min_shared_counts = pipe_vel_cfg.get("min_shared_counts", 20)
    n_jobs = pipe_vel_cfg.get("n_jobs", 4)
    
    print(f"scVelo mode: {scv_mode}")
    print(f"n_top_genes: {n_top_genes}, min_shared_counts: {min_shared_counts}")

### 2B-1. loom 파일 로드 및 adata 병합

In [ ]:
if velocity_mode == "scvelo_ready":
    # loom 파일 검색 (merged → individual 순)
    loom_search_dirs = [
        loom_dir,
        f"../../output/velocity/loom/{DATASET}",
        "../../data",
    ]
    loom_files = []
    for d in loom_search_dirs:
        if d:
            found = sorted(Path(d).rglob("*.loom"))
            if found:
                loom_files = found
                break

    if not loom_files:
        raise FileNotFoundError(
            "loom 파일을 찾을 수 없습니다.\n"
            f"검색 경로: {loom_search_dirs}\n"
            "velocyto를 실행하거나 samples.yaml의 loom_dir을 설정하세요."
        )

    print(f"loom 파일 {len(loom_files)}개 발견:")
    for f in loom_files:
        print(f"  {f}")

    # merged.loom 우선, 없으면 첫 번째 파일
    merged_loom = next((f for f in loom_files if "merged" in f.name), loom_files[0])
    ldata = scv.read(str(merged_loom), cache=True)
    print(f"\nLoom: {merged_loom}")
    print(ldata)

In [ ]:
if velocity_mode == "scvelo_ready":
    # adata + ldata 병합 (바코드 기준)
    adata_velo = scv.utils.merge(adata, ldata)
    print(f"\n병합 후: {adata_velo.shape}")
    print(f"  spliced  layer: {'spliced' in adata_velo.layers}")
    print(f"  unspliced layer: {'unspliced' in adata_velo.layers}")
    
    if adata_velo.n_obs < 100:
        print("\n⚠️  병합 후 세포 수가 적습니다. 바코드 형식을 확인하세요.")
        print("  h5ad 바코드 예시:", adata.obs_names[:3].tolist())
        print("  loom 바코드 예시:", ldata.obs_names[:3].tolist())

### 2B-2. 전처리 및 velocity 계산

In [ ]:
if velocity_mode == "scvelo_ready":
    # 전처리
    scv.pp.filter_and_normalize(
        adata_velo,
        min_shared_counts=min_shared_counts,
        n_top_genes=n_top_genes,
    )
    scv.pp.moments(adata_velo, n_pcs=30, n_neighbors=30)
    
    # Velocity 계산
    if scv_mode == "dynamical":
        print("Dynamical 모델 실행 중 (시간이 걸릴 수 있습니다)...")
        scv.tl.recover_dynamics(adata_velo, n_jobs=n_jobs)
    
    scv.tl.velocity(adata_velo, mode=scv_mode)
    scv.tl.velocity_graph(adata_velo)
    print("Velocity 계산 완료")

### 2B-3. 방향성 화살표 시각화

In [ ]:
if velocity_mode == "scvelo_ready":
    # ── Stream plot (흐름 화살표, 주 시각화) ──────────────────────────
    scv.pl.velocity_embedding_stream(
        adata_velo,
        basis="umap",
        color=groupby,
        title="RNA Velocity Stream",
        legend_loc="right margin",
        figsize=(8, 6),
        save=str(FIG_OUT_DIR / "velocity_stream.png"),
    )

In [ ]:
if velocity_mode == "scvelo_ready":
    # ── Arrow plot (개별 화살표) ───────────────────────────────────────
    scv.pl.velocity_embedding(
        adata_velo,
        basis="umap",
        arrow_length=3,
        arrow_size=2,
        color=groupby,
        title="RNA Velocity Arrows",
        figsize=(8, 6),
        save=str(FIG_OUT_DIR / "velocity_arrows.png"),
    )

In [ ]:
if velocity_mode == "scvelo_ready":
    # ── Grid plot (격자 화살표) ────────────────────────────────────────
    scv.pl.velocity_embedding_grid(
        adata_velo,
        basis="umap",
        color=groupby,
        title="RNA Velocity Grid",
        figsize=(8, 6),
        save=str(FIG_OUT_DIR / "velocity_grid.png"),
    )

### 2B-4. Latent time (RNA velocity 기반 pseudotime)

In [ ]:
if velocity_mode == "scvelo_ready":
    if scv_mode == "dynamical":
        scv.tl.latent_time(adata_velo)
        scv.pl.scatter(
            adata_velo,
            color="latent_time",
            color_map="gnuplot",
            size=80,
            title="RNA Velocity Latent Time",
            figsize=(7, 5),
            save=str(FIG_OUT_DIR / "latent_time.png"),
        )
        # pseudotime을 메인 adata에 추가
        adata.obs["velocity_latent_time"] = adata_velo.obs["latent_time"].reindex(adata.obs_names)
    else:
        print("Latent time은 dynamical 모드에서만 계산됩니다.")
        print("pipeline.yaml 에서 velocity.scvelo_mode: 'dynamical' 로 변경하세요.")

### 2B-5. Velocity confidence & speed

In [ ]:
if velocity_mode == "scvelo_ready":
    scv.tl.velocity_confidence(adata_velo)
    scv.pl.scatter(
        adata_velo,
        color=["velocity_length", "velocity_confidence"],
        cmap="coolwarm",
        perc=[5, 95],
        figsize=(12, 5),
        save=str(FIG_OUT_DIR / "velocity_confidence.png"),
    )
    print(f"평균 velocity confidence: {adata_velo.obs['velocity_confidence'].mean():.3f}")
    print(f"평균 velocity length: {adata_velo.obs['velocity_length'].mean():.3f}")

### 2B-6. Driver gene 분석 (phase portrait)

In [ ]:
if velocity_mode == "scvelo_ready":
    scv.tl.rank_velocity_genes(adata_velo, groupby=groupby, min_corr=0.3)
    
    try:
        df_drivers = scv.get_df(adata_velo, "rank_velocity_genes/names")
        print("Top velocity driver genes per group:")
        print(df_drivers.head(10))
        
        # Phase portrait (spliced vs unspliced + velocity)
        top_genes = df_drivers.values.flatten()[:6].tolist()
        top_genes = [g for g in top_genes if g in adata_velo.var_names][:6]
        if top_genes:
            scv.pl.velocity(
                adata_velo,
                top_genes,
                ncols=3,
                figsize=(14, 8),
                save=str(FIG_OUT_DIR / "phase_portrait.png"),
            )
    except Exception as e:
        print(f"Driver gene 분석 실패: {e}")

## 2C. CytoTRACE + DPT (velocity 없는 경우)

In [ ]:
if velocity_mode == "no_velocity":
    try:
        import cytotrace2
        ct_result = cytotrace2.cytotrace2(adata, slot_type="counts", species="human")
        adata.obs["cytotrace_score"] = ct_result["CytoTRACE2_Score"]
        print("CytoTRACE2 score 계산 완료")
        print("높은 점수 = 미분화 (stem-like)")
        sc.pl.umap(adata, color="cytotrace_score", color_map="RdYlBu_r", title="CytoTRACE2 Score")
    except ImportError:
        print("cytotrace2 미설치. 설치: pip install cytotrace2")
        adata.obs["cytotrace_proxy"] = adata.obs["n_genes_by_counts"]
        sc.pl.umap(adata, color="cytotrace_proxy", title="n_genes (CytoTRACE proxy)")

In [ ]:
if velocity_mode == "no_velocity":
    score_col = "cytotrace_score" if "cytotrace_score" in adata.obs.columns else "n_genes_by_counts"
    root_idx = adata.obs[score_col].idxmax()
    root_cell_idx = adata.obs_names.get_loc(root_idx)
    
    adata.uns["iroot"] = root_cell_idx
    sc.tl.diffmap(adata)
    sc.tl.dpt(adata)
    
    print(f"Root cell: {root_idx} (highest {score_col})")
    sc.pl.umap(adata, color="dpt_pseudotime", color_map="viridis", title="DPT Pseudotime")

## 3. Pseudotime에 따른 유전자 발현 변화

In [ ]:
# ============================================================
# 수동 설정: trajectory에 따라 변화하는 유전자 (관심 유전자 입력)
# ============================================================
TRAJECTORY_GENES = []  # 예: ["MKI67", "CD44", "CD3D", "EPCAM"]

# pseudotime 컬럼 자동 선택
pt_col = None
for col in ["velocity_latent_time", "dpt_pseudotime"]:
    if col in adata.obs.columns:
        pt_col = col
        break

if TRAJECTORY_GENES and pt_col:
    sc.pl.scatter(
        adata,
        x=pt_col,
        y=TRAJECTORY_GENES[0],
        color=groupby,
        title=f"{TRAJECTORY_GENES[0]} vs {pt_col}",
    )
else:
    print(f"Pseudotime column: {pt_col}")
    print("Set TRAJECTORY_GENES to visualize gene expression along pseudotime")

In [ ]:
out_path = f"{CHECKPOINT_DIR}/09_trajectory.h5ad"
save_adata(adata, out_path)
print(f"Saved: {out_path}")